### **Face Detection**

In [1]:
import cv2

# Load Haar Cascade for face detection
face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
)

# Start webcam
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break
    
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    
    faces = face_cascade.detectMultiScale(
        gray,
        scaleFactor=1.3,
        minNeighbors=5
    )

    for (x, y, w, h) in faces:
        cv2.rectangle(frame, (x,y), (x+w, y+h), (0,255,0), 2)

    cv2.imshow("Face Detection", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

### **Human Detection (HOG + SVM)**

In [2]:
import cv2

hog = cv2.HOGDescriptor()
hog.setSVMDetector(cv2.HOGDescriptor_getDefaultPeopleDetector())

cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    boxes, _ = hog.detectMultiScale(frame, winStride=(8,8))

    for (x, y, w, h) in boxes:
        cv2.rectangle(frame, (x,y), (x+w, y+h), (0,0,255), 2)

    cv2.imshow("Human Detection", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

### **Facial Feature Measurement (Using MediaPipe)**

In [3]:
import cv2
import math

# Load the pre-trained Haar Cascades
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
eye_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_eye.xml')

cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break
    
    frame = cv2.flip(frame, 1)
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # 1. Detect faces
    faces = face_cascade.detectMultiScale(gray, 1.3, 5)

    for (x, y, w, h) in faces:
        cv2.rectangle(frame, (x, y), (x+w, y+h), (255, 0, 0), 2)
        
        # Region of Interest (ROI) for eyes (only search inside the face)
        roi_gray = gray[y:y+h, x:x+w]
        roi_color = frame[y:y+h, x:x+w]
        
        # 2. Detect eyes within the face
        eyes = eye_cascade.detectMultiScale(roi_gray)
        
        # We need exactly 2 eyes to calculate distance
        if len(eyes) == 2:
            centers = []
            for (ex, ey, ew, eh) in eyes:
                # Calculate center point of the eye bounding box
                eye_center = (int(ex + ew/2), int(ey + eh/2))
                centers.append((x + eye_center[0], y + eye_center[1]))
                cv2.circle(roi_color, eye_center, 5, (0, 255, 0), -1)

            # 3. Calculate distance between the two centers
            p1, p2 = centers[0], centers[1]
            # Euclidean distance: $$d = \sqrt{(x_2-x_1)^2 + (y_2-y_1)^2}$$
            distance = math.sqrt((p2[0] - p1[0])**2 + (p2[1] - p1[1])**2)

            cv2.line(frame, p1, p2, (0, 255, 255), 2)
            cv2.putText(frame, f"Dist: {int(distance)}px", (50, 50), 
                        cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

    cv2.imshow('Haar Cascade Detection', frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()